# CopyLane 판정 인코더 — 실험 카드 6 : 품목을 입력에 붙인다 (박수진, 2026-10-08)

> Baseline = v10 (`copylane-encoder-kcbert-v10-합성O-이유O`) · 골든 `529c970556d7` · 공통 dev 749행 (지문 `4ebfcb231da703b5`)
> 규약 `docs/psj/notes/인코더_개선실험_규약_20261007.md` · 카드 원문 `docs/psj/reports/v10_문턱규칙_거짓과장_실험기록_20261007.md` 의 「카드 6」 (**돌리기 전에 올린다**)
> 🔴 **dev 만 본다. 이 노트북에는 test 채점 셀이 없다** (D-175). 🔴 **탐색이다 — 이 모델은 앱 로더에 올리지 않는다** (입력 형식이 다르다).

## 무엇을 하나

Baseline 의 유형 오인 60행 중 37행은 품목을 아는 문장이다. 화장품의 8행은 정답이 전부 의약품_오인인데 거짓_과장 등이 섰다 — 「화장품이 효능을 말하면 의약품 오인」을 품목 없이는 배우기 어렵다.
그래서 **문장 앞에 품목을 붙여** 학습한다. 학습 행 · 설정 · 손실은 Baseline 과 같다.

| | Baseline | 카드 6 |
|---|---|---|
| 입력 | `[CLS] 문장 [SEP]` | `[CLS] 품목 [SEP] 문장 [SEP]` — 품목은 「식품」 · 「건기식」 · 「화장품」 · 「모름」 |
| 품목이 없는 행 | — | 「모름」 |
| 학습 때 | — | 품목이 있는 행의 30% 를 그때그때 「모름」으로 바꿔 넣는다 (품목 없이도 돌게) |
| 길이 상한 | 128 토큰 | 136 토큰 (품목 몫만큼 늘려 문장 몫이 줄지 않게) |

## 한 모델을 두 입력으로 잰다

| 방식 | 입력 | 보는 것 |
|---|---|---|
| **(가) 품목** | 골든의 품목을 넣는다 (없으면 「모름」) | 목표 ① 품목을 아는 dev 위반 216행의 유형 오인 37 → **28행 이하** |
| **(나) 모름** | 전부 「모름」 | 목표 ② 통과 기준 네 개에 걸리지 않는다 (품목을 모르는 조건) |

채택 후보 = (가) · (나) 모두 통과 기준에 걸리지 않고 목표 ① 을 채운다. 회차(조기 종료)는 (가)의 dev PR-AUC 로 고르고, 시드 · 문턱은 방식마다 그 방식의 dev 확률로 고른다.

## 미리 아는 한계

- 과탐은 이 실험으로 재지 못한다 (dev 에 적법 광고형 문장이 없다). 조건 A 는 품목이 아니라 인정 여부에 달려 있어 풀리지 않는다.
- 학습의 건기식 행은 70% 가 양성이고 합성 문구가 전부 건기식이다 — 「건기식 = 위반」 지름길을 배울 수 있다. 섹션 7 의 품목별 음성 오판정으로 본다.
- 채택에는 팀장 판정과 앱 입력 명세 변경이 든다 (D-294 가 「제품 지위를 인코더 입력으로」를 기각했다).

## 실행 순서

1. 런타임 유형 **GPU(T4)** → 위에서부터 끝까지 실행. `golden.jsonl` 을 올린다.
2. 학습은 시드 3개 = **3번** (T4 에서 약 20분). 시드 하나가 끝날 때마다 Drive `copylane_v10_card6/` 에 기록한다 — 다시 실행하면 끝난 시드는 건너뛴다.
3. Drive 에 **방식마다 시드별 dev 행별 확률 CSV 3개 + 문턱 zip**, 그리고 모델 zip 이 남는다 (**저장소에는 올리지 않는다** · D-249).

## 1. 환경 설정 + 파일 업로드

`sympy` 충돌로 `from transformers import ...` 가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random, csv, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

from google.colab import files as colab_files
if not os.path.exists("golden.jsonl"):
    print("① golden.jsonl 선택")
    colab_files.upload()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # 재동결 10-05 (8,383행)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "런타임에 예전 파일이 남았으면 런타임을 새로 시작한다")

## 2. 스위치 — 바꿀 것은 이 셀뿐이다

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `SAVE_MODEL_ZIP` | `True` | 가장 좋은 시드의 모델 zip(약 420MB)을 Drive 에 둔다. 🔴 실험 전용 — 앱 로더가 읽지 못하게 `label_scheme.json` 을 넣지 않는다 |
| `SMOKE` | `False` | `True` = 1분짜리 점검(시드 1 · 1 epoch · 학습 300줄). 수치는 의미 없다 |
| `CAT_DROP` · `GATE` · `TARGET` | 아래 값 | 🔒 **실험 카드에 적은 값이다. 고치지 않는다** |

In [ ]:
RUN_NAME = "카드6-품목입력"
SAVE_MODEL_ZIP = True
SMOKE = False

# ── 🔒 카드 6 이 바꾸는 것 — 입력
CAT_VALUES = ["식품", "건기식", "화장품"]     # 골든 학습 분할의 품목 값. 그 밖 · 빈 칸은 「모름」
CAT_UNKNOWN = "모름"
CAT_DROP = 0.30                              # 학습 때 품목이 있는 행을 「모름」으로 바꿔 넣는 비율 (배치를 만들 때마다 새로 뽑는다)
MAX_LEN = 136                                # [CLS] 품목 [SEP] 문장 [SEP] 전체. Baseline 은 문장만 128 — 품목 몫(최대 5토큰)만큼 늘렸다
MODES = ["품목", "모름"]                     # (가) 골든의 품목을 넣는다 / (나) 전부 「모름」

# ── 학습 설정 — Baseline(v10)과 같다. 바꾸지 않는다
MODEL_NAME = "beomi/kcbert-base"
SEEDS = [42, 43, 44]
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
WEIGHT_DECAY = 0.01
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
USE_WORD_ROWS = False             # 사례집 낱말 행 제외 (Baseline 과 같다)
DEV_FRAC, DEV_SEED = 0.15, 42     # 공통 dev — 바꾸면 dev 행이 달라진다
TH_OFF = 1.01                     # 편입 대기 칸(기능성화장품_오인)의 문턱 — 꺼 둠 (D-321)
SCORE_VIEW = "v11"                # 회차 · 시드 · 문턱은 dev 749행 전부로 고른다 (규약 4절)

# ── 🔒 통과 기준 (규약 5절) — Baseline 대비 · (가) (나) 둘 다에 본다
GATE = {
    "prauc_drop": 0.02,           # B  : 6종 dev macro PR-AUC(시드 평균) 하락 허용폭
    "recall_drop": 0.02,          # G1 : 위반 탐지율 하락 허용폭
    "type_recall_drop": 0.05,     # G2 : 유형별 재현율 하락 허용폭 (dev 양성 30 이상인 유형만)
    "neg_flag_up": 1,             # G3 : 음성(D · L) 오판정이 늘어도 되는 행 수
    "adopt_prauc": 0.008,
}
# ── 🔒 겨냥한 지표와 목표 (실험 카드) — (가)에서 품목을 아는 dev 위반의 유형 오인
TARGET = {"known_mis_max": 28}
# ── Baseline(v10)의 dev 값 — 규약 1절 · 공통 채점 스크립트로 잰 값. 견줄 때만 쓴다
BASELINE = {
    "seed_prauc": [0.7167, 0.7081, 0.7165], "det": 491, "scored": 510, "hit": 431, "mis": 60, "miss": 19, "neg": 17, "neg_n": 86,
    "type_tp": {"질병_예방치료_표방": (25, 30), "건강기능식품_오인": (33, 42), "의약품_오인": (19, 37), "거짓_과장": (293, 315), "소비자_기만": (62, 98)},
    "prauc6": 0.7167, "ad_prauc6": 0.8243, "known_scored": 216, "known_mis": 37,
    "cat": {"식품": {"scored": 145, "mis": 25, "neg": (2, 3)}, "건기식": {"scored": 39, "mis": 4, "neg": (2, 62)},
            "화장품": {"scored": 32, "mis": 8, "neg": (3, 6)}, "없음": {"scored": 294, "mis": 23, "neg": (10, 15)}},
}

assert 0 <= CAT_DROP < 1 and CAT_UNKNOWN not in CAT_VALUES
if SMOKE:
    SEEDS, MAX_EPOCHS = [42], 1
NOTEBOOK = "v10-card6"
TAG = f"copylane-encoder-kcbert-v10-{RUN_NAME}"

OUT_ROOT = "/content/drive/MyDrive/copylane_v10_card6" + ("_smoke" if SMOKE else "")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(OUT_ROOT, exist_ok=True)
except Exception as e:
    OUT_ROOT = "./copylane_v10_card6_out"
    os.makedirs(OUT_ROOT, exist_ok=True)
    print("🟡 Drive 연결 안 됨 — 결과를 런타임에만 둔다(런타임이 끊기면 사라진다):", e)
print("실험:", RUN_NAME, "· 시드", SEEDS, "· 「모름」 비율", CAT_DROP, "· 결과 폴더", OUT_ROOT)

## 3. 데이터 + dev 분리 — 공통 dev (Baseline 노트북과 같은 셀 · 품목 칸만 더 읽는다)

1. 묶음 = 같은 원문에서 나온 행 (`inj:T1:hf:0:0` → `hf:0:0` · 그 밖은 `id` 의 `#` 앞).
2. 원천마다 묶음의 15% 를 dev 로 (시드 42 · 묶음 이름순 → 섞기).
3. 6종마다 dev 양성이 10건보다 적으면 남은 묶음에서 채운다.
4. dev 행 = dev 묶음의 **문장 단위 · 합성 아님** 행.

같은 골든이면 **749줄 · 지문 `4ebfcb231da703b5`** 가 나와야 한다 — v11 의 `dev_ids_v11_seed42.csv` 와 같은 행 · 같은 순서다. 다르면 멈춘다.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 모델을 고르는 6종 (D-321 결정 4)
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = SEL + NEW_CONF                       # 확정 8종 — 문턱을 dev 에서 고른다
WAITING = ["기능성화장품_오인"]              # 편입 대기 — 칸은 두고 문턱은 끈다
LABEL_LIST = CONF8 + WAITING                 # 저장 순서 (팀장 10-06)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
N_SEL, N8, N_ALL = len(SEL), len(CONF8), len(LABEL_LIST)

def norm_text(s):
    return " ".join(s.split())

def group_key(rid):
    if rid.startswith("inj:"):               # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def normalize(r):
    cond, labels = r.get("조건"), list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r["id"]), "text": r["text"], "labels": labels,
            # 조건 M · D · L 은 라벨이 적혀 있어도 양성이 아니다 (D-296 개정 2) — target 이 학습 · 채점의 정답이다
            "target": [] if cond in ("M", "D", "L") else labels,
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit"), "cat": r.get("품목")}     # 🔴 카드 6 — 품목 칸을 읽는다

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def is_inj(r):
    return r["origin"] == "injected" or r["id"].startswith("inj:")

def bucket(r):
    # 8종 기준 (run_judge_dist.py 와 같은 갈래)
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "L"
    if any(l in CONF8 for l in r["labels"]): return "scored"
    if r["labels"]: return "대기"
    return "pending" if r["cond"] in ("C", "A", "B") else "neg"

golden = load_jsonl("golden.jsonl")
all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(LABEL_LIST)
assert not unknown, f"🔴 예상 밖 라벨: {unknown}"
train_all = [r for r in all_rows if r["split"] == "train"]
_got = (len(all_rows), len(train_all), len(all_rows) - len(train_all))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")
del golden, all_rows      # 🔴 여기서부터 test 행은 메모리에도 없다 — 이 노트북은 train(→ train · dev)만 본다

# ══ dev 분리 — v11 (encoder_finetune_colab_v11_kcbert_1006) 섹션 3 과 같은 코드
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)             # 원천별 15% (묶음 단위 · 시드 고정)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # 6종마다 dev 양성이 10건보다 적으면 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
DEV_EXPECT = {"529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe": (749, "4ebfcb231da703b5")}   # 골든 sha → v11 dev (줄 수 · 지문)
if GOLDEN_SHA in DEV_EXPECT:
    if (len(dev_rows), DEV_SHA) != DEV_EXPECT[GOLDEN_SHA]:
        raise SystemExit(f"🔴 dev 가 v11 과 다르다 — {len(dev_rows)}줄 · 지문 {DEV_SHA} · 기대 {DEV_EXPECT[GOLDEN_SHA]}. 분리 코드를 건드렸는지 본다")
    print(f"✅ dev = v11 과 같은 행: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
else:
    print(f"🟡 이 골든의 v11 dev 지문을 모른다 — 대조 없이 간다: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev 에 같은 id 가 둘 이상이다"

# ── dev 행의 갈래 · 채점 방식
for r in dev_rows:
    r["bucket"] = bucket(r)
B = np.array([r["bucket"] for r in dev_rows])
IS_REASON = np.array([is_reason(r) for r in dev_rows])
VIEW = {"v11": np.ones(len(dev_rows), dtype=bool),
        "A": ~IS_REASON & ~np.isin(B, ["M", "pending"])}
SEL_MASK = VIEW[SCORE_VIEW]
BUCKETS = ["scored", "대기", "pending", "M", "D", "L", "neg"]
BUCKET_KO = {"scored": "유형 있는 위반", "대기": "편입 대기 유형만", "pending": "유형 없는 위반", "M": "조건 M", "D": "조건 D", "L": "조건 L(적법)", "neg": "그 밖 음성"}
print("\ndev 갈래 (전체 / 그중 「이유」 구역):")
for b in BUCKETS:
    if (B == b).any():
        print(f"  {BUCKET_KO[b]:12s} {int((B == b).sum()):4d} / {int(((B == b) & IS_REASON).sum()):4d}")
print(f"  합계           {len(dev_rows):4d} / {int(IS_REASON.sum()):4d}")
print(f"채점 행 — v11 방식 {int(VIEW['v11'].sum())}줄 · A 방식 {int(VIEW['A'].sum())}줄  → 이번 선택 방식: {SCORE_VIEW}")

## 4. 학습셋 · 품목

학습 행을 고르는 규칙은 Baseline 과 같다 — dev 묶음을 뺀 train 에서 dev 와 같은 문구 제거 → 조건 M 제외 → 유형 없는 위반 제외 → 낱말 행 제외 → 같은 문구 병합.

달라진 것은 **병합할 때 품목이 다르면 따로 둔다**는 것 하나다. 같은 문구라도 품목에 따라 정답이 다른 행이 있다(「디톡스」 — 식품이면 거짓_과장 · 건기식_오인, 화장품이면 의약품_오인). 그래서 학습 행이 3,964 → **3,966행**이다.

In [ ]:
def cat_of(r):
    return r["cat"] if r["cat"] in CAT_VALUES else CAT_UNKNOWN

base_pool = [r for r in train_all if r["group"] not in dev_groups]
STEPS = [("dev 묶음 뺀 train", len(base_pool))]
pool0 = [r for r in base_pool if norm_text(r["text"]) not in dev_texts];                 STEPS.append(("dev 와 같은 문구 제거", len(pool0)))
pool0 = [r for r in pool0 if r["cond"] != "M"];                                          STEPS.append(("조건 M 제외", len(pool0)))
pool0 = [r for r in pool0 if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    STEPS.append(("유형 없는 위반 제외", len(pool0)))
if not USE_WORD_ROWS:
    pool0 = [r for r in pool0 if r["unit"] != "낱말"];                                   STEPS.append(("낱말 행 제외", len(pool0)))
print(" → ".join(f"{k} {n:,}" for k, n in STEPS))
assert not any(r["cond"] == "M" for r in pool0) and not any(r["target"] for r in pool0 if r["cond"] in ("D", "L"))

merged = {}
for r in pool0:                                      # 같은 문구 병합 — 라벨 합집합. 🔴 품목이 다르면 따로 둔다 (Baseline 은 품목을 보지 않았다)
    k = (cat_of(r), norm_text(r["text"]), is_reason(r))
    if k in merged:
        merged[k]["target"] = sorted(set(merged[k]["target"]) | set(r["target"]))
    else:
        merged[k] = dict(r, target=list(r["target"]))
TRAIN_ROWS = list(merged.values())
_base_n = len({(norm_text(r["text"]), is_reason(r)) for r in pool0})          # Baseline 의 병합 규칙이면 몇 행인가
if SMOKE:
    random.Random(0).shuffle(TRAIN_ROWS); TRAIN_ROWS = TRAIN_ROWS[:300]
assert not (set(r["id"] for r in TRAIN_ROWS) & set(r["id"] for r in dev_rows)), "학습 행에 dev 행이 섞였다"
assert not ({norm_text(r["text"]) for r in TRAIN_ROWS} & dev_texts), "학습 행에 dev 와 같은 문구가 있다"

def cat_table(rows):
    t = {}
    for c in CAT_VALUES + [CAT_UNKNOWN]:
        rs = [r for r in rows if cat_of(r) == c]
        t[c] = {"행": len(rs), "양성(유형 있음)": sum(1 for r in rs if r["target"]), "음성": sum(1 for r in rs if not r["target"]),
                "합성": sum(map(is_inj, rs)), "「이유」 구역": sum(map(is_reason, rs))}
    return pd.DataFrame(t).T
TRAIN_STATS = {"학습 행": len(TRAIN_ROWS), "Baseline 병합이면": _base_n, "합성": sum(map(is_inj, TRAIN_ROWS)), "이유 구역": sum(map(is_reason, TRAIN_ROWS)),
               "품목 있는 행": sum(1 for r in TRAIN_ROWS if cat_of(r) != CAT_UNKNOWN),
               "품목별": {c: sum(1 for r in TRAIN_ROWS if cat_of(r) == c) for c in CAT_VALUES + [CAT_UNKNOWN]}}
print(f"\n■ 학습 {len(TRAIN_ROWS):,}행 (Baseline 의 병합 규칙이면 {_base_n:,}행) — 품목별" + ("   (SMOKE — 300줄로 줄였다)" if SMOKE else ""))
display(cat_table(TRAIN_ROWS))
print(f"■ dev {len(dev_rows)}행 — 품목별")
display(cat_table(dev_rows))
EXPECT_TRAIN = {"학습 행": 3966, "Baseline 병합이면": 3964, "품목별": {"식품": 919, "건기식": 1208, "화장품": 221, "모름": 1618}}     # 재동결 10-05 판 · 로컬 대조 10-08
if GOLDEN_SHA in DEV_EXPECT and not SMOKE and USE_WORD_ROWS is False:
    _got = {k: TRAIN_STATS[k] for k in EXPECT_TRAIN}
    if _got != EXPECT_TRAIN:
        raise SystemExit(f"🔴 학습 행 수가 기대와 다르다 — 받은 {_got} · 기대 {EXPECT_TRAIN}. 학습셋 코드를 건드렸는지 본다")
    print("✅ 학습 3,966행 (Baseline 3,964 + 품목이 달라 따로 둔 2행) · 품목 있는 행", TRAIN_STATS["품목 있는 행"])

## 5. 입력 · 손실 · 지표

- **입력만 다르다.** 문장쌍으로 넣는다 — 앞이 품목, 뒤가 문장. 길면 **문장 쪽만** 자른다.
- 학습 배치를 만들 때마다 품목이 있는 행의 30% 를 「모름」으로 바꾼다. dev · 예측에서는 바꾸지 않는다.
- 손실(Focal γ 1.5 · pos_weight) · 회차 선택(6종 dev macro PR-AUC) · 문턱(유형별 dev F1 최대)은 Baseline 과 같다. 회차는 **(가) 품목을 넣은 dev** 로 고른다.

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, DataCollatorWithPadding, set_seed)
from sklearn.metrics import average_precision_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

def encode_pairs(cats, texts, **kw):
    """[CLS] 품목 [SEP] 문장 [SEP] — 길면 문장 쪽만 자른다."""
    return tokenizer(list(cats), list(texts), truncation="only_second", max_length=MAX_LEN, **kw)

def to_ds(rows, train):
    """행마다 두 벌을 미리 토크나이즈한다 — 품목을 넣은 것 · 「모름」을 넣은 것. 어느 쪽을 쓸지는 배치를 만들 때 정한다."""
    texts = [r["text"] for r in rows]
    a = encode_pairs([cat_of(r) for r in rows], texts)
    u = encode_pairs([CAT_UNKNOWN] * len(rows), texts)
    return Dataset.from_dict({
        "ids_cat": a["input_ids"], "tt_cat": a["token_type_ids"], "ids_unk": u["input_ids"], "tt_unk": u["token_type_ids"],
        "labels": [multihot(r["target"]).tolist() for r in rows],
        # 「모름」으로 바꿀 확률 — 학습 행 · 품목이 있을 때만. dev 는 0 (바꾸지 않는다)
        "drop": [CAT_DROP if (train and cat_of(r) != CAT_UNKNOWN) else 0.0 for r in rows]})

class CatCollator:
    """배치를 만들 때 행마다 품목 / 「모름」 중 하나를 고른다. 🔴 Baseline 과 다른 곳은 여기다."""
    def __call__(self, feats):
        batch = []
        for f in feats:
            unk = f["drop"] > 0 and random.random() < f["drop"]
            ids, tt = (f["ids_unk"], f["tt_unk"]) if unk else (f["ids_cat"], f["tt_cat"])
            batch.append({"input_ids": ids, "token_type_ids": tt, "attention_mask": [1] * len(ids)})
        out = tokenizer.pad(batch, return_tensors="pt")
        out["labels"] = torch.tensor([f["labels"] for f in feats], dtype=torch.float32)
        return out

collator = CatCollator()

Y_DEV = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)
dev_ds = to_ds(dev_rows, train=False)          # 회차 선택은 (가) 품목을 넣은 dev 로 한다

def pos_weight_of(rows):
    Y = np.stack([multihot(r["target"]) for r in rows])
    pos = Y.sum(0); neg = len(Y) - pos
    return np.clip(np.sqrt(neg / np.clip(pos, 1e-6, None)), 1, POS_WEIGHT_CAP)

class FocalTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pw = torch.tensor(pos_weight, dtype=torch.float32)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.float()
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        loss = ((self._pw.to(logits.device) * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce).mean()
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def macro_prauc(y, p, n=N_SEL):
    # 앞 n 칸의 PR-AUC 평균 — 양성이 있는 유형만. n=6 이 모델을 고르는 기준이다
    v = [safe(average_precision_score, y[:, i], p[:, i]) for i in range(n)]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else float("nan")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = 1 / (1 + np.exp(-logits))
    return {"macro_prauc": macro_prauc(labels[SEL_MASK].astype(int), p[SEL_MASK])}

TH_GRID = np.arange(0.1, 0.9001, 0.025)
def pick_thresholds(y, p):
    th = {}
    for i, l in enumerate(CONF8):
        if y[:, i].sum() < 10:
            th[l] = 0.5
            continue
        fs = [f1_score(y[:, i], p[:, i] >= t, zero_division=0) for t in TH_GRID]
        th[l] = float(round(TH_GRID[int(np.argmax(fs))], 3))
    th[WAITING[0]] = TH_OFF
    return th

def predict_probs(mdl, rows, mode, bs=64):
    """mode = 「품목」(골든의 품목 · 없으면 「모름」) 또는 「모름」(전부 「모름」)."""
    assert mode in MODES, mode
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        part = rows[s:s + bs]
        cats = [cat_of(r) if mode == "품목" else CAT_UNKNOWN for r in part]
        enc = encode_pairs(cats, [r["text"] for r in part], padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)

print("dev 양성 (선택 방식", SCORE_VIEW, "):", {l: int(Y_DEV[SEL_MASK, i].sum()) for i, l in enumerate(LABEL_LIST)})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — CPU 로는 매우 느리다. 런타임 유형을 GPU 로 바꾼다")

_ex = next((r for r in dev_rows if cat_of(r) != CAT_UNKNOWN), dev_rows[0])
_e = encode_pairs([cat_of(_ex)], [_ex["text"]])
print("입력 예 —", " ".join(tokenizer.convert_ids_to_tokens(_e["input_ids"][0])[:14]), "…")
_lens = [len(x) for x in to_ds(TRAIN_ROWS[:2000], train=True)["ids_cat"]]
print(f"학습 앞 {len(_lens):,}행의 토큰 길이 — 최대 {max(_lens)} · 상한 {MAX_LEN} 에 닿은 행 {sum(1 for x in _lens if x >= MAX_LEN)}")
from transformers import AutoConfig
assert int(getattr(AutoConfig.from_pretrained(MODEL_NAME), "max_position_embeddings", MAX_LEN)) >= MAX_LEN, "모델이 받는 길이보다 상한이 길다"

## 6. 학습 — 시드 3개

🔒 **기준과 목표를 먼저 찍고 학습한다.** 시드 하나가 끝나면 dev 확률 두 벌((가) 품목 · (나) 모름)을 Drive 에 적는다 — 다시 실행하면 끝난 시드는 건너뛴다(`♻️`).

In [ ]:
print("🔒 통과 기준 (규약 5절) — Baseline(v10) 대비 · (가) (나) 둘 다 · 방식마다 가장 좋은 시드 · 그 시드의 dev 문턱")
print(f"  B   6종 dev macro PR-AUC(시드 평균) 하락이 {GATE['prauc_drop']} 이내   (Baseline {np.mean(BASELINE['seed_prauc']):.4f})")
print(f"  G1  위반 탐지율 하락이 {GATE['recall_drop'] * 100:.0f}%p 이내   (Baseline {BASELINE['det']}/{BASELINE['scored']})")
print(f"  G2  유형별 재현율 하락이 {GATE['type_recall_drop'] * 100:.0f}%p 이내 (dev 양성 30 이상인 유형만)")
print(f"  G3  음성(D · L) 오판정이 +{GATE['neg_flag_up']}행 이내   (Baseline {BASELINE['neg']}/{BASELINE['neg_n']})")
print("🔒 겨냥한 지표와 목표 (실험 카드)")
print(f"  ① (가) 품목을 넣어 — 품목을 아는 dev 위반 {BASELINE['known_scored']}행의 유형 오인 {BASELINE['known_mis']} → {TARGET['known_mis_max']}행 이하")
print("  ② (나) 전부 「모름」으로 — 통과 기준 네 개에 걸리지 않는다")
print("  채택 후보 = (가) · (나) 모두 통과 기준에 걸리지 않고 목표 ① 을 채운다")
print("  ※ 판정은 공통 채점 스크립트(score_encoder_dev.py)로 한다. 아래 표는 미리 보는 값이다.\n")

_ids = hashlib.sha256("|".join(r["id"] for r in TRAIN_ROWS).encode()).hexdigest()
CFG_SIG = [NOTEBOOK, GOLDEN_SHA, DEV_SHA, SCORE_VIEW, MODEL_NAME, MAX_EPOCHS, PATIENCE, BATCH, LR, WARMUP, WEIGHT_DECAY, MAX_LEN,
           FOCAL_GAMMA, POS_WEIGHT_CAP, USE_WORD_ROWS, SMOKE, CAT_VALUES, CAT_UNKNOWN, CAT_DROP, _ids]
def sig_of(seed):
    return hashlib.sha256(json.dumps(CFG_SIG + [seed], ensure_ascii=False).encode()).hexdigest()[:16]

def train_seed(seed, get_ds, pw):
    keep, local = f"{OUT_ROOT}/{TAG}/seed{seed}", f"./runs/{TAG}/seed{seed}"
    done = f"{keep}/done.json"
    if os.path.exists(done) and all(os.path.exists(f"{keep}/dev_p_{m}.npy") for m in MODES):
        res = json.load(open(done, encoding="utf-8"))
        if res.get("sig") == sig_of(seed):
            res["dev_p"] = {m: np.load(f"{keep}/dev_p_{m}.npy") for m in MODES}
            print(f"♻️ seed {seed}: 이미 끝남 — 건너뜀 (best epoch {res['best_epoch']})")
            return res
        print(f"seed {seed}: 저장된 기록이 지금 설정과 다르다 — 다시 학습한다")
    t0 = time.time()
    train_ds = get_ds()
    set_seed(seed)                                   # 파이썬 난수도 여기서 고정된다 — 「모름」 바꾸기가 시드마다 같다
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_ALL, problem_type="multi_label_classification",
        id2label=dict(enumerate(LABEL_LIST)), label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"./runs/{TAG}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed, dataloader_num_workers=0,
        fp16=torch.cuda.is_available())
    trainer = FocalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, data_collator=collator,
                           compute_metrics=compute_metrics, pos_weight=pw,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = {m: predict_probs(trainer.model, dev_rows, m) for m in MODES}
    shutil.rmtree(local, ignore_errors=True)
    trainer.save_model(local)
    shutil.rmtree(f"./runs/{TAG}/ckpt{seed}", ignore_errors=True)
    res = {"sig": sig_of(seed), "seed": seed, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
           "curve": [[int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)] for h in hist], "minutes": round((time.time() - t0) / 60, 1)}
    os.makedirs(keep, exist_ok=True)
    for m in MODES:
        np.save(f"{keep}/dev_p_{m}.npy", dev_p[m])
    shutil.copy(f"{local}/config.json", f"{keep}/config.json")
    json.dump(res, open(done, "w", encoding="utf-8"), ensure_ascii=False)       # 🔴 done.json 을 맨 끝에 쓴다 — 있으면 이 시드는 끝난 것이다
    res["dev_p"] = dev_p
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

def scheme_of(mode, R):
    M_, s = R[mode], R[mode]["best_seed"]
    return {
        "label_list": LABEL_LIST, "label_thresholds": M_["thresholds"],
        "confirmed_labels": CONF8, "waiting_labels": WAITING, "legacy_confirmed_labels": SEL,
        "backbone": MODEL_NAME, "notebook": f"{NOTEBOOK} (박수진 · 실험 카드 6 — 품목을 입력에 붙인다)",
        "experiment": f"{TAG}-{mode}", "baseline": "copylane-encoder-kcbert-v10-합성O-이유O",
        "experiment_only": True, "app_loader": "🔴 올리지 않는다 — 입력이 [CLS] 품목 [SEP] 문장 [SEP] 이다. 문장만 넣으면 다른 수가 나온다",
        "input_format": "pair: (품목, 문장) · truncation=only_second", "category_values": CAT_VALUES, "category_unknown": CAT_UNKNOWN,
        "category_drop": CAT_DROP, "eval_mode": mode, "eval_mode_note": "품목 = 골든의 품목을 넣는다(없으면 모름) / 모름 = 전부 모름",
        "max_len": MAX_LEN, "max_len_note": "문장쌍 전체 길이다 (Baseline 은 문장만 128)",
        "seed": s, "best_epoch": R["seeds"][s]["best_epoch"], "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS),
        "train_rows": len(TRAIN_ROWS), "train_category_rows": TRAIN_STATS["품목별"],
        "train_rule": "조건 M 제외 · D · L 음성 · 유형 없는 위반 제외 · 낱말 행 제외 · 같은 문구 병합(품목이 다르면 따로)",
        "dev_rule": f"공통 dev — 원천별 묶음 {DEV_FRAC:.0%} + 6종 보충 · 시드 {DEV_SEED} · 문장 단위 · 합성 아님",
        "dev_sha": DEV_SHA, "dev_rows": len(dev_rows),
        "threshold_note": "이 방식의 가장 좋은 시드의 dev F1 최대 · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
        "epoch_note": "회차(조기 종료)는 (가) 품목을 넣은 dev 의 6종 macro PR-AUC 로 골랐다",
        "thresholds_by_seed": {str(k): v["thresholds"] for k, v in M_["seeds"].items()},
        "dev_macro_prauc6_by_seed": {str(k): v["prauc6"] for k, v in M_["seeds"].items()},
        "final_test": False, "test_note": "이 노트북은 test 를 채점하지 않는다 (D-175)", "smoke": SMOKE,
    }

def export_run(R):
    """방식마다 시드별 dev 행별 확률 CSV · 문턱 zip, 그리고 (스위치) (가)의 가장 좋은 시드의 모델 zip."""
    out = []
    for m in MODES:
        for sd in SEEDS:
            path = f"{OUT_ROOT}/{TAG}_{m}_seed{sd}_dev행별확률.csv"
            with open(path, "w", encoding="utf-8-sig", newline="") as f:          # 문구는 넣지 않는다 (id · 갈래 · 확률만)
                w = csv.writer(f)
                w.writerow(["id", "조건", "bucket", "이유구역", "품목"] + [f"p_{l}" for l in LABEL_LIST])
                for r, row in zip(dev_rows, R["seeds"][sd]["dev_p"][m]):
                    w.writerow([r["id"], r["cond"] or "", r["bucket"], int(is_reason(r)), cat_of(r)] + [f"{x:.6f}" for x in row])
            out.append(os.path.basename(path))
        with zipfile.ZipFile(f"{OUT_ROOT}/{TAG}_{m}_scheme.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.writestr("label_scheme.json", json.dumps(scheme_of(m, R), ensure_ascii=False, indent=2))
        out.append(f"{TAG}_{m}_scheme.zip")
    s = R["품목"]["best_seed"]
    local = f"./runs/{TAG}/seed{s}"
    if not SAVE_MODEL_ZIP:
        pass
    elif not os.path.exists(f"{local}/config.json"):
        if os.path.exists(f"{OUT_ROOT}/{TAG}_실험전용.zip") and R.get("zip_seed") == s:
            out.append(f"{TAG}_실험전용.zip (앞서 저장한 것)")
        else:
            print(f"  🟡 가장 좋은 시드 {s} 의 가중치가 이 런타임에 없다(앞 세션에서 학습) — 모델 zip 은 못 만든다. "
                  f"필요하면 Drive 의 {TAG} 폴더를 지우고 다시 돌린다")
    else:
        sdir = f"./export/{TAG}"
        shutil.rmtree(sdir, ignore_errors=True)
        m_ = AutoModelForSequenceClassification.from_pretrained(local)
        assert [m_.config.id2label[i] for i in range(m_.config.num_labels)] == LABEL_LIST, "저장 칸 순서가 다르다 (D-220)"
        m_.save_pretrained(sdir, safe_serialization=True)
        tokenizer.save_pretrained(sdir)
        # 🔴 label_scheme.json 을 넣지 않는다 — 앱 로더(app/encoder.py)가 이 폴더를 읽지 못하게 한다. 입력 형식이 달라 문장만 넣으면 틀린 수가 나온다
        json.dump({m: scheme_of(m, R) for m in MODES}, open(f"{sdir}/label_scheme.experiment.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        open(f"{sdir}/README_실험전용.txt", "w", encoding="utf-8").write(
            "실험 카드 6 (품목을 입력에 붙인다) 의 모델이다. 앱 로더에 올리지 않는다.\n"
            "입력은 [CLS] 품목 [SEP] 문장 [SEP] 이다 (품목 = 식품 · 건기식 · 화장품 · 모름). 문턱은 label_scheme.experiment.json 에 방식별로 있다.\n")
        assert not os.path.exists(f"{sdir}/label_scheme.json")
        try:
            zp = shutil.make_archive(f"./export/{TAG}_실험전용", "zip", sdir)
            shutil.copy(zp, f"{OUT_ROOT}/{TAG}_실험전용.zip")
            json.dump({"zip_seed": s}, open(f"{OUT_ROOT}/{TAG}/zip_seed.json", "w"))
            out.append(f"{TAG}_실험전용.zip")
            os.remove(zp); shutil.rmtree(sdir, ignore_errors=True)
        except Exception as e:
            print(f"  🟡 모델 zip 저장 안 됨 (Drive 용량?) — {e}")
        del m_
    return out

T_ALL = time.time()
pw = pos_weight_of(TRAIN_ROWS)
print(f"{'=' * 22} {RUN_NAME} · 학습 {len(TRAIN_ROWS):,}행 · 품목 있는 행 {TRAIN_STATS['품목 있는 행']:,} (그중 {CAT_DROP:.0%} 를 그때그때 「모름」으로) {'=' * 22}")
print("pos_weight:", {l: round(float(pw[i]), 2) for i, l in enumerate(LABEL_LIST)})
_cache, _seeds = {}, {}
def get_ds():                                     # 학습할 시드가 있을 때만 토크나이즈한다
    if "ds" not in _cache:
        _cache["ds"] = to_ds(TRAIN_ROWS, train=True)
    return _cache["ds"]
for seed in SEEDS:
    res = train_seed(seed, get_ds, pw)
    _seeds[seed] = res
    print(f"  seed {seed}: best epoch {res['best_epoch']}/{res['epochs_run']} · 곡선((가) dev) {res['curve']} · {res['minutes']}분 · 6종 PR-AUC "
          + " / ".join(f"({'가' if m == '품목' else '나'}) {m} {macro_prauc(Y_DEV, res['dev_p'][m], N_SEL):.4f}" for m in MODES))
_zs = f"{OUT_ROOT}/{TAG}/zip_seed.json"
RESULTS = {"seeds": _seeds, "zip_seed": json.load(open(_zs))["zip_seed"] if os.path.exists(_zs) else None}
for m in MODES:
    per = {}
    for seed in SEEDS:
        p = _seeds[seed]["dev_p"][m]
        per[seed] = {"prauc6": macro_prauc(Y_DEV, p, N_SEL), "prauc8": macro_prauc(Y_DEV, p, N8), "thresholds": pick_thresholds(Y_DEV, p)}
    best = max(SEEDS, key=lambda s: per[s]["prauc6"])
    RESULTS[m] = {"seeds": per, "best_seed": best, "thresholds": per[best]["thresholds"],
                  "prauc6_mean": float(np.mean([per[s]["prauc6"] for s in SEEDS])), "prauc6_std": float(np.std([per[s]["prauc6"] for s in SEEDS]))}
    print(f"[{m}] 6종 dev macro PR-AUC {RESULTS[m]['prauc6_mean']:.4f} ± {RESULTS[m]['prauc6_std']:.4f} · 가장 좋은 시드 {best}")
    print("  문턱:", {l: RESULTS[m]["thresholds"][l] for l in CONF8})
print("저장:", export_run(RESULTS))
for s in SEEDS:                                   # 디스크 — (가)의 가장 좋은 시드의 가중치만 남긴다
    if s != RESULTS["품목"]["best_seed"]:
        shutil.rmtree(f"./runs/{TAG}/seed{s}", ignore_errors=True)
print(f"\n학습 끝 — {round((time.time() - T_ALL) / 60, 1)}분")

## 7. 결과 — 미리 보기

- **표 1** 방식마다 가장 좋은 시드 · 그 시드의 문턱으로 잰 값. Baseline 칸은 규약의 값이다.
- **표 2** 품목별 — 유형 오인과 음성 오판정. 목표 ① 은 「품목을 아는 행」 줄이다. 건기식의 음성 오판정이 늘면 지름길을 배운 것이다.
- **표 3** 유형별 재현율 · 정밀도.
- **표 4** 통과 기준과 목표.

읽는 법 — 「유형 오인」 = 유형 있는 위반에 후보는 섰는데 정답 유형이 후보에 없다. (가)와 (나)의 차이가 품목이 실제로 해 주는 일이다.

In [ ]:
SC = B == "scored"
NEG = np.isin(B, ["D", "L", "neg"])
AD = ~IS_REASON & ~np.isin(B, ["M", "pending"])          # 광고 문구 줄
Y8 = Y_DEV[:, :N8] == 1
F_IDX = LABEL2ID["거짓_과장"]
CAT = np.array([cat_of(r) for r in dev_rows])
KNOWN = CAT != CAT_UNKNOWN

def measure(p, th):
    fl = p[:, :N8] >= np.array([th[l] for l in CONF8])
    any_, hit = fl.any(1), (fl & Y8).any(1)
    mis = any_ & ~hit
    out = {"scored": int(SC.sum()), "det": int(any_[SC].sum()), "hit": int(hit[SC].sum()), "mis": int(mis[SC].sum()),
           "miss": int((~any_)[SC].sum()), "neg": int(any_[NEG].sum()), "neg_n": int(NEG.sum()),
           "D": (int(any_[B == "D"].sum()), int((B == "D").sum())), "L": (int(any_[B == "L"].sum()), int((B == "L").sum())),
           "false_cand": int(fl[:, F_IDX].sum()), "mis_only_false": int((mis & SC & fl[:, F_IDX] & (fl.sum(1) == 1)).sum()),
           "known_scored": int((SC & KNOWN).sum()), "known_mis": int((mis & SC & KNOWN).sum()),
           "prauc6": macro_prauc(Y_DEV, p), "prauc8": macro_prauc(Y_DEV, p, N8), "ad_prauc6": macro_prauc(Y_DEV[AD], p[AD])}
    out["type"] = {l: {"n": int(Y8[:, i].sum()), "tp": int((fl[:, i] & Y8[:, i]).sum()), "pp": int(fl[:, i].sum())} for i, l in enumerate(CONF8)}
    out["cat"] = {}
    for c in CAT_VALUES + [CAT_UNKNOWN]:
        k = CAT == c
        out["cat"][c] = {"scored": int((SC & k).sum()), "det": int((any_ & SC & k).sum()), "hit": int((hit & SC & k).sum()), "mis": int((mis & SC & k).sum()),
                         "neg": (int((any_ & NEG & k).sum()), int((NEG & k).sum()))}
    return out

def rate(k, n):
    return "—" if n == 0 else f"{k}/{n} ({k / n * 100:.1f}%)"

M = {}
for m in MODES:
    R = RESULTS[m]
    M[m] = measure(RESULTS["seeds"][R["best_seed"]]["dev_p"][m], R["thresholds"])
    M[m]["per_seed"] = {s: measure(RESULTS["seeds"][s]["dev_p"][m], R["seeds"][s]["thresholds"]) for s in SEEDS}
b = BASELINE
b_mean, b_std = float(np.mean(b["seed_prauc"])), float(np.std(b["seed_prauc"]))
NAME = {"품목": "(가) 품목을 넣어", "모름": "(나) 전부 「모름」"}

# ── 표 1
_t = {"Baseline v10 (규약 값)": {
    "6종 PR-AUC 시드 평균": f"{b_mean:.4f} ± {b_std:.4f}", "선택 시드": "42", "선택 시드 PR-AUC (dev 전체 · 광고 문구 줄)": f"{b['prauc6']:.3f} · {b['ad_prauc6']:.3f}",
    "위반 탐지 ↑": rate(b["det"], b["scored"]), "정답 유형 적중 ↑": rate(b["hit"], b["scored"]), "유형 오인 ↓": rate(b["mis"], b["scored"]),
    "— 품목을 아는 행 ↓": rate(b["known_mis"], b["known_scored"]), "놓침 ↓": rate(b["miss"], b["scored"]), "음성(D · L) 오판정 ↓": f"{b['neg']}/{b['neg_n']}"}}
for m in MODES:
    R, x = RESULTS[m], M[m]
    _t[NAME[m]] = {
        "6종 PR-AUC 시드 평균": f"{R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f}", "선택 시드": f"{R['best_seed']} (epoch {RESULTS['seeds'][R['best_seed']]['best_epoch']})",
        "선택 시드 PR-AUC (dev 전체 · 광고 문구 줄)": f"{x['prauc6']:.3f} · {x['ad_prauc6']:.3f}",
        "위반 탐지 ↑": rate(x["det"], x["scored"]), "정답 유형 적중 ↑": rate(x["hit"], x["scored"]), "유형 오인 ↓": rate(x["mis"], x["scored"]),
        "— 품목을 아는 행 ↓": rate(x["known_mis"], x["known_scored"]), "놓침 ↓": rate(x["miss"], x["scored"]),
        "음성(D · L) 오판정 ↓": f"{x['neg']}/{x['neg_n']} (D {x['D'][0]} · L {x['L'][0]})"}
print(f"■ 표 1 · dev {len(dev_rows)}행 — 방식마다 가장 좋은 시드 · 그 시드의 문턱 (유형 있는 위반 {int(SC.sum())} · 그중 품목을 아는 행 {int((SC & KNOWN).sum())} · 음성 {int(NEG.sum())})")
display(pd.DataFrame(_t).T)
print("시드별 (자기 문턱) — 품목을 아는 행의 유형 오인 · 전체 유형 오인 · 위반 탐지 · 음성 오판정:")
for m in MODES:
    print(f"  {NAME[m]}: " + "  /  ".join(f"시드 {s} → {v['known_mis']} · {v['mis']} · {v['det']} · {v['neg']}" for s, v in M[m]["per_seed"].items()))

# ── 표 2 · 품목별
_t = {}
for c in CAT_VALUES + [CAT_UNKNOWN]:
    bc = b["cat"]["없음" if c == CAT_UNKNOWN else c]
    _t[c + (" (품목 없음)" if c == CAT_UNKNOWN else "")] = {
        "유형 있는 위반": M["품목"]["cat"][c]["scored"], "유형 오인 — Baseline": bc["mis"],
        **{f"유형 오인 — {NAME[m]}": M[m]["cat"][c]["mis"] for m in MODES},
        "음성 오판정 — Baseline": f"{bc['neg'][0]}/{bc['neg'][1]}",
        **{f"음성 오판정 — {NAME[m]}": "{}/{}".format(*M[m]["cat"][c]["neg"]) for m in MODES}}
_t["품목을 아는 행 (목표 ①)"] = {"유형 있는 위반": M["품목"]["known_scored"], "유형 오인 — Baseline": b["known_mis"],
                               **{f"유형 오인 — {NAME[m]}": M[m]["known_mis"] for m in MODES}, "음성 오판정 — Baseline": "",
                               **{f"음성 오판정 — {NAME[m]}": "" for m in MODES}}
print("\n■ 표 2 · 품목별 유형 오인 · 음성(D · L) 오판정")
display(pd.DataFrame(_t).T)

# ── 표 3 · 유형별
_t = {}
for i, l in enumerate(CONF8):
    n = int(Y8[:, i].sum())
    _t[l] = {"dev 양성": f"{n}" + (" (측정 불가 n<30)" if n < 30 else ""),
             "Baseline 재현율": "—" if l not in b["type_tp"] else f"{b['type_tp'][l][0] / b['type_tp'][l][1] * 100:.1f}%"}
    for m in MODES:
        s = M[m]["type"][l]
        _t[l][NAME[m]] = f"R {s['tp'] / max(s['n'], 1) * 100:.1f}% · P {s['tp'] / max(s['pp'], 1) * 100:.1f}% · τ {RESULTS[m]['thresholds'][l]}"
print("\n■ 표 3 · 유형별 재현율 R · 정밀도 P · 문턱 τ")
display(pd.DataFrame(_t).T)

# ── 표 4 · 통과 기준 · 목표
VERDICT, _t = {}, {}
for m in MODES:
    R, x = RESULTS[m], M[m]
    d_pr = R["prauc6_mean"] - b_mean
    d_rec = x["det"] / x["scored"] - b["det"] / b["scored"]
    d_type = {l: x["type"][l]["tp"] / x["type"][l]["n"] - tp / n for l, (tp, n) in b["type_tp"].items() if x["type"][l]["n"] >= 30}
    worst = min(d_type, key=d_type.get) if d_type else None
    d_neg = x["neg"] - b["neg"]
    ok = {"B": d_pr >= -GATE["prauc_drop"], "G1": d_rec >= -GATE["recall_drop"] - 1e-12,
          "G2": (d_type[worst] >= -GATE["type_recall_drop"] - 1e-12) if worst else True, "G3": d_neg <= GATE["neg_flag_up"]}
    fails = [k for k, v in ok.items() if not v]
    VERDICT[m] = {"fails": fails, "d_prauc": d_pr, "d_recall": d_rec, "d_neg": d_neg, "worst_type": worst, "d_worst": d_type.get(worst)}
    _t[NAME[m]] = {"B · PR-AUC 시드 평균": f"{R['prauc6_mean']:.4f} ({d_pr:+.4f})", "G1 · 위반 탐지율": f"{rate(x['det'], x['scored'])} ({d_rec * 100:+.1f}%p)",
                   "G2 · 유형별 최대 하락": "—" if worst is None else f"{worst} {d_type[worst] * 100:+.1f}%p",
                   "G3 · 음성 오판정": f"{x['neg']}/{x['neg_n']} ({d_neg:+d}행)", "통과 기준": "통과" if not fails else "탈락 (" + " · ".join(fails) + ")"}
goal1 = M["품목"]["known_mis"] <= TARGET["known_mis_max"]
goal2 = not VERDICT["모름"]["fails"]
if VERDICT["품목"]["fails"] or VERDICT["모름"]["fails"]:
    overall = "탈락 — " + " / ".join(f"{NAME[m]} {' · '.join(VERDICT[m]['fails'])}" for m in MODES if VERDICT[m]["fails"])
elif goal1:
    overall = "채택 후보"
else:
    overall = "통과 · 목표 ① 미달"
VERDICT["overall"] = {"verdict": overall, "goal1": bool(goal1), "goal2": bool(goal2)}
print("\n■ 표 4 · 통과 기준 · 목표 — Baseline(v10 · 규약 값) 대비")
display(pd.DataFrame(_t).T)
print(f"목표 ① (가) 품목을 아는 행의 유형 오인 {M['품목']['known_mis']}행 (Baseline {b['known_mis']} · 목표 {TARGET['known_mis_max']} 이하) → {'✅ 충족' if goal1 else '❌ 미달'}")
print(f"목표 ② (나) 통과 기준 → {'✅ 걸리지 않음' if goal2 else '❌ ' + ' · '.join(VERDICT['모름']['fails'])}")
print(f"품목이 해 준 일 — 품목을 아는 행의 유형 오인: (나) 모름 {M['모름']['known_mis']}행 → (가) 품목 {M['품목']['known_mis']}행")
print(f"미리 본 판정: {overall}")
print("※ 미리 보는 값이다. 판정은 공통 채점 스크립트로 하고(섹션 8), 채택 후보가 나와도 팀장 판정 · 앱 입력 명세 변경이 든다 (D-294). test 는 팀장 판정 뒤 1회다 (D-175).")
if SMOKE:
    print("⚠️ SMOKE 실행 — 위 수치는 의미 없다")

SUMMARY = {"notebook": NOTEBOOK, "golden_sha256": GOLDEN_SHA, "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "gate": GATE, "target": TARGET,
           "baseline": BASELINE, "smoke": SMOKE, "train": TRAIN_STATS, "verdict": VERDICT, "category_drop": CAT_DROP, "max_len": MAX_LEN,
           "seeds": {str(s): {x: v[x] for x in ("best_epoch", "epochs_run", "curve", "minutes")} for s, v in RESULTS["seeds"].items()},
           "modes": {m: {"best_seed": RESULTS[m]["best_seed"], "prauc6_mean": RESULTS[m]["prauc6_mean"], "prauc6_std": RESULTS[m]["prauc6_std"],
                         "thresholds": RESULTS[m]["thresholds"], "seeds": {str(s): v for s, v in RESULTS[m]["seeds"].items()},
                         "dev": {x: v for x, v in M[m].items() if x != "per_seed"},
                         "dev_by_seed": {str(s): v for s, v in M[m]["per_seed"].items()}} for m in MODES}}
json.dump(SUMMARY, open(f"{OUT_ROOT}/results_v10_card6.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
print("\n결과 폴더:", OUT_ROOT)
for f in sorted(os.listdir(OUT_ROOT)):
    if os.path.isfile(f"{OUT_ROOT}/{f}"):
        print(f"  {f:72s} {os.path.getsize(f'{OUT_ROOT}/{f}') / 1e6:8.1f} MB")

## 8. 다음 단계

**1) 받을 파일** — Drive `copylane_v10_card6/` 에서 **CSV 6개(방식 2 × 시드 3) + `_scheme.zip` 2개**를 받아 저장소 밖(`Downloads\copylane_v10\`)에 둔다. 모델 zip 은 채택 후보일 때만 받는다.

**2) 공통 채점 (로컬 · 판정은 여기서 한다)** — 방식마다 한 번씩 돌린다. 아래는 (가) 품목. (나)는 파일 이름의 `_품목_` 을 `_모름_` 으로, `--target` 과 `--out-md` 를 바꾼다.

```
uv run python docs\psj\e2e_prototype\score_encoder_dev.py --probs "<카드6-품목입력_품목_seed42.csv>" "<…_품목_seed43.csv>" "<…_품목_seed44.csv>" --scheme "<카드6-품목입력_품목_scheme.zip>" --baseline-probs "<v10-합성O-이유O_dev행별확률.csv>" --baseline-scheme "<v10-합성O-이유O_scheme.zip>" --baseline-seed-prauc 0.7167 0.7081 0.7165 --target "(가) 품목을 넣어 — 품목을 아는 dev 위반 216행의 유형 오인 37 → 28행 이하" --out-md docs\psj\reports\v10_카드6_품목_dev채점_20261008.md
```

집계표의 「품목별」 표에서 식품 · 건기식 · 화장품의 「F1 유형 오인」을 더한 값이 목표 ① 의 수다.

**3) 무엇을 보고 고르나**

- (가) · (나) 가운데 하나라도 통과 기준에 걸리면 **탈락**이다. 「문턱을 맞춘 비교」로 품질 탓인지 문턱 탓인지 적는다.
- 둘 다 걸리지 않고 목표 ① 을 채우면 **채택 후보** → 팀장 판정(입력 명세 · D-294) → 다른 한 사람이 재현.
- 목표(28행) · 「모름」 비율(30%)은 결과를 본 뒤 고치지 않는다. 다른 값은 새 카드로 본다.

**4) 올리지 않는 것** — 모델 zip · 행별 확률 CSV · 이 노트북의 실행 출력은 저장소에 올리지 않는다 (D-249). 🔴 이 모델을 `COPYLANE_MODEL_DIR` 에 두지 않는다.